# Ablation notebook generator

This notebook is the **sole editable source** for the six run notebooks:

`ablation_baseline.ipynb`, `ablation_optical_only.ipynb`, `ablation_sar_only.ipynb`, `ablation_no_slope.ipynb`, `ablation_no_slic.ipynb`, and `ablation_no_bonferroni.ipynb`.

Edit `SHARED_CONFIG` or `VARIANTS` below, then run all cells. Set `CHECK_ONLY = True` to validate committed notebooks without rewriting them. The generator embeds the full pipeline in every run notebook, so the run notebooks never import repository code.

In [1]:
import hashlib
import json
import re
from pathlib import Path

SHARED_CONFIG = {'raw_repo': 'sasudo2/landslides', 'raw_commit_sha': 'REPLACE_WITH_40_CHARACTER_RAW_COMMIT_SHA', 'output_repo': 'sasudo2/landslides', 'output_revision': 'main', 'study_root': 'ablations/study_01', 'source_csv_name': 'landslides_from_2018_to_2026.csv', 'cohort_start': 900, 'cohort_stop': 1000, 'workers': 4}
VARIANTS = {'baseline': {'modality': 'fused', 'slope': True, 'slic': True, 'bonferroni': True}, 'optical_only': {'modality': 'optical', 'slope': True, 'slic': True, 'bonferroni': True}, 'sar_only': {'modality': 'sar', 'slope': True, 'slic': True, 'bonferroni': True}, 'no_slope': {'modality': 'fused', 'slope': False, 'slic': True, 'bonferroni': True}, 'no_slic': {'modality': 'fused', 'slope': True, 'slic': False, 'bonferroni': True}, 'no_bonferroni': {'modality': 'fused', 'slope': True, 'slic': True, 'bonferroni': False}}
ALL_VARIANTS = tuple(VARIANTS)
WRITE_NOTEBOOKS = True
CHECK_ONLY = False
NOTEBOOK_VERSION = "ablation-notebook-v1"

PIPELINE_CELLS = [
r'''def norm01(x):
    a = np.nanmin(x)
    b = np.nanmax(x)
    if not np.isfinite(a) or not np.isfinite(b) or b <= a:
        return np.zeros_like(x, dtype=np.float32)
    return ((x - a) / (b - a)).astype(np.float32)


def wmean_cov(x, weights):
    weights = weights / np.sum(weights)
    mean = np.sum(x * weights[:, None], axis=0)
    centered = x - mean
    return mean, (centered * weights[:, None]).T @ centered


def cca_axes(sxx, syy, sxy, reg=None):
    if reg is None:
        reg = SETTINGS["irmad_reg"]
    lx = np.linalg.cholesky(sxx + reg * np.eye(sxx.shape[0]))
    ly = np.linalg.cholesky(syy + reg * np.eye(syy.shape[0]))
    inv_lx = np.linalg.inv(lx)
    inv_ly = np.linalg.inv(ly)
    u, _, vt = np.linalg.svd(inv_lx @ sxy @ inv_ly.T, full_matrices=False)
    return inv_lx.T @ u, inv_ly.T @ vt.T


def irmad(x, y, max_iters=None, tol=None):
    if max_iters is None:
        max_iters = SETTINGS["max_iters"]
    if tol is None:
        tol = SETTINGS["irmad_tol"]
    n, p = x.shape
    weights = np.ones(n, dtype=np.float64)
    previous = None
    chi = None
    for _ in range(max_iters):
        mx, sxx = wmean_cov(x, weights)
        my, syy = wmean_cov(y, weights)
        xc = x - mx
        yc = y - my
        sxy = (xc * (weights / np.sum(weights))[:, None]).T @ yc
        a, b = cca_axes(sxx, syy, sxy)
        mad = xc @ a[:, :p] - yc @ b[:, :p]
        std = np.std(mad, axis=0) + 1e-6
        chi = np.sum((mad / std[None, :]) ** 2, axis=1)
        no_change_p = 1.0 - chi2.cdf(chi, df=p)
        if previous is not None and np.mean(np.abs(no_change_p - previous)) < tol:
            break
        previous = no_change_p
        weights = np.clip(no_change_p, 1e-6, 1.0)
    if chi is None:
        raise ValueError("IR-MAD requires at least one iteration")
    return chi, chi2.cdf(chi, df=p)


def build_target_grid(ref_path, scale_m=None):
    if scale_m is None:
        scale_m = SETTINGS["scale_m"]
    with rasterio.open(ref_path) as src:
        _, bottom, _, top = transform_bounds(src.crs, TARGET_CRS, *src.bounds)
        mean_lat_rad = np.deg2rad((bottom + top) / 2.0)
        m_per_deg_lon = SETTINGS["m_per_deg_lat"] * max(np.cos(mean_lat_rad), 1e-6)
        transform, width, height = calculate_default_transform(
            src.crs,
            TARGET_CRS,
            src.width,
            src.height,
            *src.bounds,
            resolution=(scale_m / m_per_deg_lon, scale_m / SETTINGS["m_per_deg_lat"]),
        )
    if width is None or height is None:
        raise ValueError(f"Could not determine analysis-grid dimensions for {ref_path}")
    return transform, int(width), int(height), mean_lat_rad


def reproject_bands(path, indexes, transform, shape):
    out = np.zeros((len(indexes), *shape), dtype=np.float32)
    with rasterio.open(path) as src:
        for i, index in enumerate(indexes):
            reproject(
                source=rasterio.band(src, index),
                destination=out[i],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=TARGET_CRS,
                resampling=Resampling.bilinear,
            )
    if not np.all(np.isfinite(out)):
        raise ValueError(f"Non-finite raster values in {path}")
    return out


def _looks_like_alpha_band(band):
    values = band[np.isfinite(band)]
    if values.size < 100:
        return False
    low, high = np.percentile(values, [1, 99])
    return bool(high - low < 2.0 or len(np.unique(values)) <= 8)


def select_rgb_band_indexes(path):
    with rasterio.open(path) as src:
        if src.count >= 4:
            return [1, 2, 3] if _looks_like_alpha_band(src.read(4)) else [3, 2, 1]
        if src.count >= 3:
            return [1, 2, 3]
        raise ValueError(f"Optical raster has too few bands ({src.count}) in {path}")


def load_optical(before_path, after_path, transform, shape):
    before_indexes = select_rgb_band_indexes(before_path)
    after_indexes = select_rgb_band_indexes(after_path)
    n_bands = min(len(before_indexes), len(after_indexes), 3)
    before_indexes = before_indexes[:n_bands]
    after_indexes = after_indexes[:n_bands]
    before = reproject_bands(before_path, before_indexes, transform, shape)
    after = reproject_bands(after_path, after_indexes, transform, shape)
    valid = np.all(before > 0, axis=0) & np.all(after > 0, axis=0)
    meta = {
        "before_rgb_band_indexes": before_indexes,
        "after_rgb_band_indexes": after_indexes,
    }
    return before, after, valid, meta


def load_slope(path, transform, shape):
    return reproject_bands(path, [1], transform, shape)[0]


def load_sar(pre_path, post_path, transform, shape):
    return (
        reproject_bands(pre_path, [1, 2], transform, shape),
        reproject_bands(post_path, [1, 2], transform, shape),
    )


def crop_native(path, indexes, bbox):
    with rasterio.open(path) as src:
        left, bottom, right, top = transform_bounds(TARGET_CRS, src.crs, *bbox)
        win = src.window(left, bottom, right, top)
        c0 = max(0, int(np.floor(win.col_off)))
        r0 = max(0, int(np.floor(win.row_off)))
        c1 = min(src.width, int(np.ceil(win.col_off + win.width)))
        r1 = min(src.height, int(np.ceil(win.row_off + win.height)))
        if c1 <= c0 or r1 <= r0:
            raise ValueError(f"Candidate chip does not overlap {path}")
        win = Window.from_slices((r0, r1), (c0, c1))
        return src.read(indexes, window=win).astype(np.float32), src.window_transform(win), src.crs


def write_chip(path, array, transform, crs):
    data = array[None, ...] if array.ndim == 2 else array
    with rasterio.open(
        path,
        "w",
        driver="GTiff",
        dtype="float32",
        count=data.shape[0],
        height=data.shape[1],
        width=data.shape[2],
        crs=crs,
        transform=transform,
    ) as dst:
        dst.write(data.astype(np.float32))''',
r'''def change_evidence(before, after, optical):
    bands, height, width = before.shape
    if optical:
        valid = np.all(before > 0, axis=0) & np.all(after > 0, axis=0)
    else:
        valid = np.all(before != 0, axis=0) & np.all(after != 0, axis=0)
    if np.count_nonzero(valid) <= 100:
        raise ValueError("Selected modality has <=100 valid paired pixels; cannot fit IR-MAD")
    x = np.moveaxis(before, 0, -1)[valid]
    y = np.moveaxis(after, 0, -1)[valid]
    chi_values, confidence_values = irmad(x, y)
    if not np.all(np.isfinite(chi_values)):
        raise ValueError("IR-MAD produced non-finite change evidence")
    confidence = np.zeros((height, width), dtype=np.float32)
    chi = np.zeros((height, width), dtype=np.float64)
    df = np.zeros((height, width), dtype=np.float64)
    confidence[valid] = confidence_values
    chi[valid] = chi_values
    df[valid] = bands
    return confidence, chi, df


def segmentation_features(confidence, post, slope):
    channels = [confidence[..., None]]
    channels.append(np.stack([norm01(band) for band in post], axis=-1))
    if ABLATION_SWITCHES["slope"]:
        channels.append(norm01(slope)[..., None])
    return np.concatenate(channels, axis=-1)


def aggregate_segments(confidence, chi, df, terrain, segments):
    ids, inverse = np.unique(segments, return_inverse=True)
    inverse = inverse.reshape(segments.shape)
    active = inverse[terrain]
    counts = np.bincount(active, minlength=len(ids))
    conf_sum = np.bincount(active, weights=confidence[terrain], minlength=len(ids))
    chi_sum = np.bincount(active, weights=chi[terrain], minlength=len(ids))
    df_sum = np.bincount(active, weights=df[terrain], minlength=len(ids))
    conf_mean = np.divide(conf_sum, counts, out=np.zeros(len(ids)), where=counts > 0)
    z = np.divide(chi_sum - df_sum, np.sqrt(2.0 * df_sum), out=np.zeros(len(ids)), where=df_sum > 0)
    return conf_mean[inverse].astype(np.float32), z[inverse].astype(np.float32), len(ids)


def detection_mask(confidence, chi, df, terrain, features, px_area):
    if ABLATION_SWITCHES["slic"]:
        pixels_per_segment = max(1, int(SETTINGS["target_superpixel_m"] ** 2 / max(px_area, 1e-6)))
        n_segments = max(SETTINGS["slic_min_segments"], int(confidence.size / pixels_per_segment))
        segments = slic(
            features,
            n_segments=n_segments,
            compactness=SETTINGS["slic_compactness"],
            start_label=1,
            channel_axis=-1,
        )
        seg_conf, z, n_tests = aggregate_segments(confidence, chi, df, terrain, segments)
    else:
        seg_conf = confidence
        z = np.divide(chi - df, np.sqrt(2.0 * df), out=np.zeros_like(chi), where=df > 0)
        n_tests = max(confidence.size, 1)
    alpha = SETTINGS["alpha"] / n_tests if ABLATION_SWITCHES["bonferroni"] else SETTINGS["alpha"]
    z_crit = norm.ppf(1.0 - alpha)
    valid_conf = seg_conf[terrain]
    if valid_conf.size == 0 or np.max(valid_conf) <= 0:
        return np.zeros_like(terrain, dtype=bool), n_tests, float(z_crit)
    high = np.percentile(valid_conf, SETTINGS["high_percentile"])
    low = min(np.percentile(valid_conf, SETTINGS["low_percentile"]), high)
    mask = apply_hysteresis_threshold(seg_conf, low, high)
    mask = mask & terrain & (z >= z_crit)
    mask = remove_small_objects(mask, min_size=SETTINGS["min_mask_objects_px"])
    return mask, n_tests, float(z_crit)


def detect(paths):
    transform, width, height, latitude = build_target_grid(paths["before"])
    shape = (height, width)
    slope = load_slope(paths["slope"], transform, shape)
    evidence = []
    post_features = None
    optical_meta = None
    if ABLATION_SWITCHES["modality"] in ("optical", "fused"):
        before, after, valid, optical_meta = load_optical(paths["before"], paths["after"], transform, shape)
        evidence.append(change_evidence(before, after, True))
        post_features = after
    if ABLATION_SWITCHES["modality"] in ("sar", "fused"):
        pre, post = load_sar(paths["sar_pre"], paths["sar_post"], transform, shape)
        evidence.append(change_evidence(pre, post, False))
        if ABLATION_SWITCHES["modality"] == "sar":
            post_features = post
    if post_features is None:
        raise ValueError(f"Unknown modality: {ABLATION_SWITCHES['modality']}")
    confidence = np.power(
        np.prod(np.stack([item[0] for item in evidence], axis=0), axis=0),
        1.0 / len(evidence),
    ).astype(np.float32)
    chi = np.sum([item[1] for item in evidence], axis=0)
    df = np.sum([item[2] for item in evidence], axis=0)
    if ABLATION_SWITCHES["slope"]:
        terrain = slope >= SETTINGS["min_slope_deg"]
    else:
        terrain = np.ones(shape, dtype=bool)
    confidence *= terrain
    features = segmentation_features(confidence, post_features, slope) if ABLATION_SWITCHES["slic"] else None
    m_per_deg_lon = SETTINGS["m_per_deg_lat"] * max(np.cos(latitude), 1e-6)
    px_area = abs(transform.a * transform.e) * m_per_deg_lon * SETTINGS["m_per_deg_lat"]
    mask, n_tests, z_crit = detection_mask(confidence, chi, df, terrain, features, px_area)
    candidates = []
    for region in regionprops(label(mask), intensity_image=confidence):
        area = region.area * px_area
        if not SETTINGS["min_blob_area_m2"] <= area <= SETTINGS["max_blob_area_m2"]:
            continue
        elongation = (region.major_axis_length or 1) / max(region.minor_axis_length or 1, 1e-6)
        if elongation > SETTINGS["max_elongation"]:
            continue
        r0, c0, r1, c1 = region.bbox
        lon_min, lat_max = xy(transform, r0, c0, offset="ul")
        lon_max, lat_min = xy(transform, r1, c1, offset="ul")
        severity = float(region.mean_intensity)
        candidates.append({
            "area_m2": float(area),
            "elongation": float(elongation),
            "severity": severity,
            "score": float(area * severity),
            "lon_min": float(lon_min),
            "lat_min": float(lat_min),
            "lon_max": float(lon_max),
            "lat_max": float(lat_max),
            "bbox_px": [int(r0), int(c0), int(r1), int(c1)],
        })
    candidates.sort(key=lambda item: item["score"], reverse=True)
    return {
        "candidates": candidates[:SETTINGS["max_candidates"]],
        "mask": mask,
        "slope": slope,
        "transform": transform,
        "px_area": px_area,
        "n_tests": n_tests,
        "z_crit": z_crit,
        "optical_meta": optical_meta,
    }


def export_candidates(incident_id, paths, result, folder):
    height, width = result["mask"].shape
    transform = result["transform"]
    pad = max(1, int(round(SETTINGS["chip_pad_m"] / max(np.sqrt(result["px_area"]), 1e-6))))
    optical_meta = result["optical_meta"]
    if optical_meta is None:
        optical_meta = {
            "before_rgb_band_indexes": select_rgb_band_indexes(paths["before"]),
            "after_rgb_band_indexes": select_rgb_band_indexes(paths["after"]),
        }
    rows = []
    for candidate_id, candidate in enumerate(result["candidates"], 1):
        row = {key: value for key, value in candidate.items() if key != "bbox_px"}
        row.update(incident_id=incident_id, candidate_id=candidate_id)
        r0, c0, r1, c1 = candidate["bbox_px"]
        r0 = max(0, r0 - pad)
        c0 = max(0, c0 - pad)
        r1 = min(height, r1 + pad)
        c1 = min(width, c1 + pad)
        chip_tf = window_transform(Window.from_slices((r0, r1), (c0, c1)), transform)
        left, top = chip_tf * (0, 0)
        right, bottom = chip_tf * (c1 - c0, r1 - r0)
        bbox = (left, bottom, right, top)
        chip_dir = folder / f"candidate_{candidate_id}"
        chip_dir.mkdir(parents=True)
        specs = {
            "before": crop_native(paths["before"], optical_meta["before_rgb_band_indexes"], bbox),
            "after": crop_native(paths["after"], optical_meta["after_rgb_band_indexes"], bbox),
            "slope": (result["slope"][r0:r1, c0:c1], chip_tf, TARGET_CRS),
            "mask": (result["mask"][r0:r1, c0:c1], chip_tf, TARGET_CRS),
        }
        for chip_type, (array, chip_transform, chip_crs) in specs.items():
            write_chip(
                chip_dir / f"incident_{incident_id}_candidate_{candidate_id}_{chip_type}.tif",
                array,
                chip_transform,
                chip_crs,
            )
        rows.append(row)
    return rows''',
r'''TRANSIENT_STATUS_CODES = {408, 409, 412, 425, 429, 500, 502, 503, 504}


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def write_json(path, data):
    Path(path).write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def write_csv(path, fields, rows):
    with Path(path).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def canonical_json(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False)


def _status_code(exc):
    response = getattr(exc, "response", None)
    return getattr(response, "status_code", None)


def is_transient_hub_error(exc):
    status = _status_code(exc)
    if status in TRANSIENT_STATUS_CODES:
        return True
    if isinstance(exc, (ConnectionError, TimeoutError)):
        return True
    message = str(exc).lower()
    markers = (
        "timeout", "timed out", "temporarily", "connection reset", "connection aborted",
        "rate limit", "too many requests", "server error", "bad gateway",
        "service unavailable", "gateway timeout",
    )
    return any(marker in message for marker in markers) or status in TRANSIENT_STATUS_CODES


def retry_hub(action, description, attempts=5, base_delay=2.0):
    for attempt in range(1, attempts + 1):
        try:
            return action()
        except Exception as exc:
            if attempt >= attempts or not is_transient_hub_error(exc):
                raise
            delay = min(base_delay ** attempt, 60.0) + random.uniform(0.0, 1.0)
            print(f"{description} failed on attempt {attempt}/{attempts}: {exc}; retrying in {delay:.1f}s")
            time.sleep(delay)
    raise RuntimeError(f"Retry loop exhausted for {description}")


def repo_files(repo, revision, prefix):
    def list_tree():
        return list(API.list_repo_tree(
            repo_id=repo,
            repo_type="dataset",
            revision=revision,
            path_in_repo=prefix,
            recursive=True,
        ))

    try:
        entries = retry_hub(list_tree, f"list files under {repo}@{revision}/{prefix}")
    except Exception as exc:
        status = _status_code(exc)
        message = str(exc).lower()
        if status == 404 or "not found" in message or "entry not found" in message:
            return set()
        raise
    files = set()
    for entry in entries:
        path = getattr(entry, "path", None)
        if path and (getattr(entry, "type", None) == "file" or hasattr(entry, "blob_id")):
            files.add(path)
    return files


def resolve_immutable_sha(repo, configured):
    configured = configured.strip().lower()
    if not re.fullmatch(r"[0-9a-f]{40}", configured):
        raise ValueError(
            "Configure the raw commit as a full 40-character SHA; branches such as main are rejected"
        )
    info = retry_hub(
        lambda: API.repo_info(repo_id=repo, repo_type="dataset", revision=configured),
        f"resolve {repo}@{configured}",
    )
    resolved = str(getattr(info, "sha", "")).lower()
    if resolved != configured:
        raise ValueError(f"Raw revision resolved to {resolved or 'unknown'}, expected {configured}")
    return resolved


def raw_suffixes_by_incident(files):
    pattern = re.compile(rf"^{re.escape(RAW_ROOT)}/incident_(\d+)/incident_\1_(.+)$")
    grouped = {}
    for filename in files:
        match = pattern.match(filename)
        if match:
            grouped.setdefault(int(match.group(1)), set()).add(match.group(2))
    return grouped


def verify_cohort_inputs(incident_ids, grouped):
    missing = {}
    before_by_incident = {}
    mandatory = set(MANDATORY_RAW_SUFFIXES)
    for incident_id in incident_ids:
        suffixes = grouped.get(incident_id, set())
        absent = sorted(mandatory - suffixes)
        before = next((suffix for suffix in BEFORE_RAW_SUFFIXES if suffix in suffixes), None)
        if absent or before is None:
            missing[str(incident_id)] = {
                "missing_mandatory": absent,
                "missing_before": before is None,
            }
        else:
            before_by_incident[incident_id] = before
    if missing:
        raise RuntimeError(
            "Selected incidents lack required raw inputs at the pinned commit: "
            + json.dumps(missing, sort_keys=True)
        )
    return before_by_incident


def fetch_paths(incident_id, before_by_incident):
    prefix = f"{RAW_ROOT}/incident_{incident_id}/incident_{incident_id}_"
    suffix_by_key = {
        "before": before_by_incident[incident_id],
        "after": "after.tif",
        "slope": "slope.tif",
        "sar_pre": "sar_pre.tif",
        "sar_post": "sar_post.tif",
    }
    paths = {}
    for key, suffix in suffix_by_key.items():
        paths[key] = retry_hub(
            lambda key=key, suffix=suffix: hf_hub_download(
                repo_id=RAW_REPO,
                repo_type="dataset",
                revision=study["raw_commit_sha"],
                filename=f"{prefix}{suffix}",
                token=HF_TOKEN,
            ),
            f"download {prefix}{suffix}",
        )
    return paths


def package_versions():
    versions = {}
    for package in PACKAGE_NAMES:
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            versions[package] = "missing"
    return versions


def notebook_file_sha256():
    candidates = []
    vsc_path = globals().get("__vsc_ipynb_file__")
    if vsc_path:
        candidates.append(Path(vsc_path))
    candidates.extend([
        Path("/kaggle/working/repo") / f"ablation_{VARIANT_NAME}.ipynb",
        Path.cwd() / f"ablation_{VARIANT_NAME}.ipynb",
    ])
    for path in candidates:
        if path.is_file():
            return hashlib.sha256(path.read_bytes()).hexdigest()
    return None


def run_environment():
    return {
        "python": platform.python_version(),
        "implementation": platform.python_implementation(),
        "platform": platform.platform(),
        "machine": platform.machine(),
        "processor": platform.processor(),
        "cpu_count": os.cpu_count(),
        "hostname": socket.gethostname(),
        "kaggle_kernel_run_type": os.environ.get("KAGGLE_KERNEL_RUN_TYPE"),
    }


def locate_source_csv():
    explicit = Path("/kaggle/input/landslide-reproted") / SOURCE_CSV_NAME
    if explicit.is_file():
        return explicit
    matches = []
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        matches.extend(sorted(input_root.rglob(SOURCE_CSV_NAME)))
    local = Path.cwd() / SOURCE_CSV_NAME
    if local.is_file():
        matches.append(local)
    if not matches:
        raise FileNotFoundError(
            f"Attach the Kaggle dataset 'landslide-reproted' containing {SOURCE_CSV_NAME}"
        )
    preferred = [path for path in matches if "reproted" in str(path).lower()]
    return preferred[0] if len(preferred) == 1 else matches[0]


def build_study(study_root, incident_ids, before_by_incident, raw_sha, versions, source_csv_path):
    return {
        "study_schema_version": STUDY_SCHEMA_VERSION,
        "output_schema_version": OUTPUT_SCHEMA_VERSION,
        "variant": VARIANT_NAME,
        "switches": dict(ABLATION_SWITCHES),
        "raw_repo": RAW_REPO,
        "raw_commit_sha": raw_sha,
        "source_csv_name": SOURCE_CSV_NAME,
        "source_csv_sha256": hashlib.sha256(Path(source_csv_path).read_bytes()).hexdigest(),
        "cohort": {
            "declaration": f"df.iloc[{COHORT_START}:{COHORT_STOP}]",
            "start": COHORT_START,
            "stop": COHORT_STOP,
            "count": len(incident_ids),
        },
        "incident_ids": list(incident_ids),
        "required_raw_inputs": dict(REQUIRED_RAW_INPUTS),
        "before_file_by_incident": {str(key): value for key, value in before_by_incident.items()},
        "settings": dict(SETTINGS),
        "package_versions": versions,
        "pipeline_sha256": PIPELINE_SHA256,
        "template_sha256": TEMPLATE_SHA256,
        "generated_notebook_sha256": GENERATED_NOTEBOOK_SHA256,
        "generated_notebook_file_sha256": notebook_file_sha256(),
        "generated_at": utc_now(),
        "run_environment": run_environment(),
        "study_root": study_root,
    }


def fetch_sibling_manifests(study_root, output_files):
    manifests = {}
    for other in ALL_VARIANTS:
        if other == VARIANT_NAME:
            continue
        manifest_path = f"{study_root}/{other}/study.json"
        if manifest_path not in output_files:
            continue
        local_path = retry_hub(
            lambda manifest_path=manifest_path: hf_hub_download(
                repo_id=OUTPUT_REPO,
                repo_type="dataset",
                revision=OUTPUT_REVISION,
                filename=manifest_path,
                token=HF_TOKEN,
            ),
            f"read sibling manifest {manifest_path}",
        )
        manifests[other] = json.loads(Path(local_path).read_text(encoding="utf-8"))
    return manifests


def check_sibling_identity(study, manifests):
    conflicts = []
    for other, sibling in manifests.items():
        if sibling.get("variant") != other:
            conflicts.append(f"{other}: manifest variant is {sibling.get('variant')!r}")
        for key in SHARED_IDENTITY_KEYS:
            if key not in sibling:
                conflicts.append(f"{other}: missing shared field {key}")
            elif canonical_json(sibling[key]) != canonical_json(study[key]):
                conflicts.append(f"{other}: shared field {key} differs")
    if conflicts:
        raise RuntimeError(
            "Existing sibling study manifests conflict with this run; use a new STUDY_ROOT: "
            + "; ".join(conflicts)
        )''',
]

SETUP_MARKDOWN = r'''## Kaggle setup

1. In **Settings -> Internet**, turn internet **on**.
2. Attach the `landslide-reproted` dataset so `landslides_from_2018_to_2026.csv` appears under `/kaggle/input/`.
3. Add the Kaggle secret `huggingface_token` with read/write access to the output dataset repository.
4. Confirm `RAW_COMMIT_SHA`, `OUTPUT_REPO`, `OUTPUT_REVISION`, and `STUDY_ROOT` below. All six run notebooks must use the same values.
5. Run all cells. Do not run two copies of the *same* variant at once.'''
PIP_INSTALL_CELL = r'''%pip install -q "numpy>=1.24" "pandas>=2.0" "scipy>=1.10" "rasterio>=1.3" "scikit-image>=0.19,<2" "huggingface_hub>=0.23"'''
CONFIG_TEMPLATE = r'''from __future__ import annotations

import csv
import hashlib
import importlib.metadata
import json
import os
import platform
import random
import re
import shutil
import socket
import tempfile
import time
import traceback
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd
import rasterio
from huggingface_hub import CommitOperationAdd, HfApi, hf_hub_download
from rasterio.transform import xy
from rasterio.warp import Resampling, calculate_default_transform, reproject, transform_bounds
from rasterio.windows import Window, transform as window_transform
from scipy.stats import chi2, norm
from skimage.filters import apply_hysteresis_threshold
from skimage.measure import label, regionprops
from skimage.morphology import remove_small_objects
from skimage.segmentation import slic

VARIANT_NAME = __VARIANT_NAME__
ABLATION_SWITCHES = __ABLATION_SWITCHES__
ALL_VARIANTS = __ALL_VARIANTS__
EXPECTED_VARIANTS = __EXPECTED_VARIANTS__

RAW_REPO = __RAW_REPO__
RAW_COMMIT_SHA = __RAW_COMMIT_SHA__
OUTPUT_REPO = __OUTPUT_REPO__
OUTPUT_REVISION = __OUTPUT_REVISION__
STUDY_ROOT = __STUDY_ROOT__
SOURCE_CSV_NAME = __SOURCE_CSV_NAME__
COHORT_START = __COHORT_START__
COHORT_STOP = __COHORT_STOP__
WORKERS = __WORKERS__
UPLOAD_BATCH_FILES = 100
RAW_ROOT = "raw_images/raw_incidents"
TARGET_CRS = "EPSG:4326"
STUDY_SCHEMA_VERSION = "ablation-study-v1"
OUTPUT_SCHEMA_VERSION = "ablation-output-v1"
STATUS_FIELDS = ["incident_id", "status", "n_candidates", "error"]
META_FIELDS = [
    "incident_id", "candidate_id", "area_m2", "elongation", "severity", "score",
    "lon_min", "lat_min", "lon_max", "lat_max",
]
PACKAGE_NAMES = ("numpy", "pandas", "scipy", "rasterio", "scikit-image", "huggingface_hub")
MANDATORY_RAW_SUFFIXES = ("after.tif", "slope.tif", "sar_pre.tif", "sar_post.tif")
BEFORE_RAW_SUFFIXES = ("planet_before.tif", "gee_before.tif")
REQUIRED_RAW_INPUTS = {
    "mandatory": list(MANDATORY_RAW_SUFFIXES),
    "before_any_of": list(BEFORE_RAW_SUFFIXES),
}
SHARED_IDENTITY_KEYS = (
    "study_schema_version",
    "output_schema_version",
    "raw_repo",
    "raw_commit_sha",
    "source_csv_name",
    "source_csv_sha256",
    "cohort",
    "incident_ids",
    "required_raw_inputs",
    "before_file_by_incident",
    "settings",
    "package_versions",
    "pipeline_sha256",
    "template_sha256",
)

SETTINGS = {
    "scale_m": 10.0,
    "min_slope_deg": 20.0,
    "target_superpixel_m": 30.0,
    "high_percentile": 99.7,
    "low_percentile": 93.0,
    "alpha": 0.05,
    "max_iters": 20,
    "irmad_tol": 1e-3,
    "irmad_reg": 1e-6,
    "min_blob_area_m2": 3000.0,
    "max_blob_area_m2": 2000000.0,
    "max_elongation": 5.0,
    "max_candidates": 100,
    "chip_pad_m": 300.0,
    "slic_compactness": 0.2,
    "min_mask_objects_px": 5,
    "slic_min_segments": 50,
    "m_per_deg_lat": 111320.0,
}

PIPELINE_SHA256 = __PIPELINE_SHA256__
TEMPLATE_SHA256 = __TEMPLATE_SHA256__
GENERATED_NOTEBOOK_SHA256 = __GENERATED_NOTEBOOK_SHA256__


def resolve_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError as exc:
        raise RuntimeError("Set HF_TOKEN or run on Kaggle with the huggingface_token secret attached") from exc
    try:
        return UserSecretsClient().get_secret("huggingface_token")
    except Exception as exc:
        raise RuntimeError("Attach the Kaggle secret 'huggingface_token' with read/write access") from exc


HF_TOKEN = resolve_hf_token()
API = HfApi(token=HF_TOKEN)'''
PREFLIGHT_CELL = r'''def validate_study_root(value):
    path = PurePosixPath(value)
    if (
        not path.parts
        or path.is_absolute()
        or value != path.as_posix()
        or any(part in (".", "..") or part.startswith(".") for part in path.parts)
        or path.parts[0] in ("candidates", "raw_images")
    ):
        raise ValueError("STUDY_ROOT must be a new relative folder such as ablations/study_01")
    return value


def reserve_variant(study_root, study):
    LOCAL_VARIANT.mkdir(parents=True, exist_ok=False)
    run = {
        "variant": VARIANT_NAME,
        "switches": dict(ABLATION_SWITCHES),
        "study_root": study_root,
        "output_repo": OUTPUT_REPO,
        "output_revision": OUTPUT_REVISION,
        "workers": WORKERS,
        "started_at": utc_now(),
        "status": "running",
        "generated_notebook_sha256": GENERATED_NOTEBOOK_SHA256,
        "study_identity_sha256": hashlib.sha256(
            canonical_json({key: study[key] for key in SHARED_IDENTITY_KEYS}).encode("utf-8")
        ).hexdigest(),
    }
    write_json(LOCAL_VARIANT / "study.json", study)
    write_json(LOCAL_VARIANT / "run.json", run)
    operations = [
        CommitOperationAdd(
            path_in_repo=f"{study_root}/{VARIANT_NAME}/study.json",
            path_or_fileobj=str(LOCAL_VARIANT / "study.json"),
        ),
        CommitOperationAdd(
            path_in_repo=f"{study_root}/{VARIANT_NAME}/run.json",
            path_or_fileobj=str(LOCAL_VARIANT / "run.json"),
        ),
    ]
    retry_hub(
        lambda: API.create_commit(
            repo_id=OUTPUT_REPO,
            repo_type="dataset",
            revision=OUTPUT_REVISION,
            operations=operations,
            commit_message=f"Reserve ablation {VARIANT_NAME}",
        ),
        f"reserve {study_root}/{VARIANT_NAME}",
    )
    return run


if VARIANT_NAME not in EXPECTED_VARIANTS:
    raise ValueError(f"Unknown variant name: {VARIANT_NAME}")
if ABLATION_SWITCHES != EXPECTED_VARIANTS[VARIANT_NAME]:
    raise ValueError(
        f"Variant switches for {VARIANT_NAME} do not match the canonical configuration"
    )
if WORKERS < 1:
    raise ValueError("WORKERS must be at least 1")
if COHORT_START != 900 or COHORT_STOP != 1000:
    raise ValueError("This study requires the fixed cohort df.iloc[900:1000]")

study_root = validate_study_root(STUDY_ROOT)
source_csv_path = locate_source_csv()
source_df = pd.read_csv(source_csv_path)
if "id" not in source_df.columns:
    raise ValueError(f"{SOURCE_CSV_NAME} has no 'id' column")
if len(source_df) < COHORT_STOP:
    raise ValueError(f"{SOURCE_CSV_NAME} has {len(source_df)} rows, need at least {COHORT_STOP}")
cohort_df = source_df.iloc[COHORT_START:COHORT_STOP]
if len(cohort_df) != COHORT_STOP - COHORT_START:
    raise ValueError("Cohort selection did not return exactly 100 rows")
incident_ids = [int(value) for value in cohort_df["id"].tolist()]
if len(set(incident_ids)) != len(incident_ids):
    raise ValueError("Cohort contains duplicate incident IDs")

raw_sha = resolve_immutable_sha(RAW_REPO, RAW_COMMIT_SHA)
raw_files = repo_files(RAW_REPO, raw_sha, RAW_ROOT)
grouped_raw = raw_suffixes_by_incident(raw_files)
before_by_incident = verify_cohort_inputs(incident_ids, grouped_raw)
versions = package_versions()
study = build_study(study_root, incident_ids, before_by_incident, raw_sha, versions, source_csv_path)

output_files = repo_files(OUTPUT_REPO, OUTPUT_REVISION, study_root)
own_prefix = f"{study_root}/{VARIANT_NAME}/"
existing_own = sorted(path for path in output_files if path.startswith(own_prefix))
if existing_own:
    raise FileExistsError(
        f"Remote variant folder already exists: {own_prefix} "
        f"(for example {existing_own[0]}); use a new STUDY_ROOT"
    )
manifests = fetch_sibling_manifests(study_root, output_files)
check_sibling_identity(study, manifests)

LOCAL_ROOT = Path(tempfile.mkdtemp(prefix=f"ablation_{VARIANT_NAME}_"))
LOCAL_VARIANT = LOCAL_ROOT / VARIANT_NAME
RUN = reserve_variant(study_root, study)
print(
    json.dumps(
        {
            "variant": VARIANT_NAME,
            "switches": ABLATION_SWITCHES,
            "cohort_size": len(incident_ids),
            "raw_commit_sha": raw_sha,
            "sibling_manifests_checked": sorted(manifests),
            "local_work_dir": str(LOCAL_VARIANT),
        },
        sort_keys=True,
    )
)'''
RUN_CELL = r'''def process_incident(incident_id):
    paths = fetch_paths(incident_id, before_by_incident)
    result = detect(paths)
    incident_folder = LOCAL_VARIANT / f"incident_{incident_id}"
    incident_folder.mkdir(parents=True, exist_ok=True)
    rows = export_candidates(incident_id, paths, result, incident_folder)
    diagnostic = {
        "incident_id": incident_id,
        "n_tests": result["n_tests"],
        "z_crit": result["z_crit"],
        "mask_pixels": int(np.count_nonzero(result["mask"])),
        "n_candidates": len(rows),
    }
    return rows, diagnostic, incident_folder


records = {}
rows_by_id = {}
diagnostics = {}
pending_ops = []
pending_ids = []


def flush_pending():
    if not pending_ops:
        return True
    operations = list(pending_ops)
    incident_batch = list(pending_ids)
    succeeded = True
    try:
        retry_hub(
            lambda: API.create_commit(
                repo_id=OUTPUT_REPO,
                repo_type="dataset",
                revision=OUTPUT_REVISION,
                operations=operations,
                commit_message=f"Ablation {VARIANT_NAME}: chips for {len(incident_batch)} incident(s)",
            ),
            f"upload chips for {VARIANT_NAME} incidents {incident_batch}",
        )
    except Exception as exc:
        succeeded = False
        for incident_id in incident_batch:
            record = records.setdefault(
                incident_id,
                {"incident_id": incident_id, "status": "ok", "n_candidates": 0, "error": ""},
            )
            record.update(status="failed", error=f"Upload {type(exc).__name__}: {exc}")
            rows_by_id.pop(incident_id, None)
            diagnostics.pop(incident_id, None)
        print(f"Batch upload failed for {incident_batch}: {exc}")
    finally:
        for incident_id in incident_batch:
            shutil.rmtree(LOCAL_VARIANT / f"incident_{incident_id}", ignore_errors=True)
        pending_ops.clear()
        pending_ids.clear()
    return succeeded


with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futures = {executor.submit(process_incident, incident_id): incident_id for incident_id in incident_ids}
    for future in as_completed(futures):
        incident_id = futures[future]
        record = {"incident_id": incident_id, "status": "ok", "n_candidates": 0, "error": ""}
        records[incident_id] = record
        try:
            rows, diagnostic, incident_folder = future.result()
        except Exception as exc:
            record.update(status="failed", error=f"{type(exc).__name__}: {exc}")
            traceback.print_exc()
            shutil.rmtree(LOCAL_VARIANT / f"incident_{incident_id}", ignore_errors=True)
            print(f"{VARIANT_NAME} incident_{incident_id} failed: {exc}")
            continue
        record["n_candidates"] = len(rows)
        rows_by_id[incident_id] = rows
        diagnostics[incident_id] = diagnostic
        print(f"{VARIANT_NAME} incident_{incident_id}: {len(rows)} candidates")
        if rows:
            prefix = f"{study_root}/{VARIANT_NAME}/incident_{incident_id}"
            pending_ops.extend(
                CommitOperationAdd(
                    path_in_repo=f"{prefix}/{path.relative_to(incident_folder).as_posix()}",
                    path_or_fileobj=str(path),
                )
                for path in sorted(incident_folder.rglob("*.tif"))
            )
            pending_ids.append(incident_id)
            if len(pending_ops) >= UPLOAD_BATCH_FILES:
                flush_pending()
        else:
            shutil.rmtree(incident_folder, ignore_errors=True)

flush_pending()

records_list = [records[incident_id] for incident_id in incident_ids]
metadata_rows = []
for incident_id in incident_ids:
    metadata_rows.extend(rows_by_id.get(incident_id, []))
metadata_rows.sort(key=lambda row: (int(row["incident_id"]), int(row["candidate_id"])))
diagnostics_list = sorted(
    (diagnostics[incident_id] for incident_id in incident_ids if incident_id in diagnostics),
    key=lambda item: item["incident_id"],
)
write_csv(LOCAL_VARIANT / "candidate_status.csv", STATUS_FIELDS, records_list)
write_csv(LOCAL_VARIANT / "candidate_metadata.csv", META_FIELDS, metadata_rows)
write_json(LOCAL_VARIANT / "diagnostics.json", diagnostics_list)

n_failed = sum(record["status"] == "failed" for record in records_list)
n_ok = len(records_list) - n_failed
n_candidates = sum(record["n_candidates"] for record in records_list if record["status"] == "ok")
n_incidents_with_candidates = sum(
    record["status"] == "ok" and record["n_candidates"] > 0 for record in records_list
)
finished_at = utc_now()
summary = {
    "variant": VARIANT_NAME,
    "switches": dict(ABLATION_SWITCHES),
    "study_root": study_root,
    "cohort_size": len(incident_ids),
    "n_incidents": len(records_list),
    "n_ok": n_ok,
    "n_failed": n_failed,
    "n_candidates": n_candidates,
    "n_incidents_with_candidates": n_incidents_with_candidates,
    "candidates_per_successful_incident": (n_candidates / n_ok) if n_ok else 0.0,
    "complete": bool(n_failed == 0 and len(records_list) == len(incident_ids)),
    "raw_commit_sha": study["raw_commit_sha"],
    "pipeline_sha256": PIPELINE_SHA256,
    "template_sha256": TEMPLATE_SHA256,
    "finished_at": finished_at,
}
write_json(LOCAL_VARIANT / "summary.json", summary)
RUN.update({
    "finished_at": finished_at,
    "status": "complete" if summary["complete"] else "failed",
    "n_incidents": len(records_list),
    "n_ok": n_ok,
    "n_failed": n_failed,
    "n_candidates": n_candidates,
    "n_incidents_with_candidates": n_incidents_with_candidates,
    "failed_incident_ids": [record["incident_id"] for record in records_list if record["status"] == "failed"],
})
write_json(LOCAL_VARIANT / "run.json", RUN)

retry_hub(
    lambda: API.upload_folder(
        repo_id=OUTPUT_REPO,
        repo_type="dataset",
        revision=OUTPUT_REVISION,
        folder_path=str(LOCAL_VARIANT),
        path_in_repo=f"{study_root}/{VARIANT_NAME}",
        allow_patterns=["*.csv", "*.json"],
        commit_message=f"Summarize ablation {VARIANT_NAME}",
    ),
    f"upload summaries for {VARIANT_NAME}",
)

print(json.dumps(summary, sort_keys=True))
if n_failed:
    raise RuntimeError(
        f"{n_failed} incident(s) failed in {VARIANT_NAME}; inspect candidate_status.csv "
        "and rerun the study with a new STUDY_ROOT"
    )'''

GENERATED_SHA_PLACEHOLDER = "__GENERATED_NOTEBOOK_SHA256__"


def notebook_dir():
    current = Path.cwd()
    for candidate in (current, *current.parents):
        if (candidate / "candidate_detection.ipynb").is_file():
            return candidate
    return current


NOTEBOOK_DIR = notebook_dir()


def pipeline_sha256():
    return hashlib.sha256("\n\n".join(PIPELINE_CELLS).encode("utf-8")).hexdigest()


def template_sha256():
    parts = [
        "TITLE_TEMPLATE",
        SETUP_MARKDOWN,
        PIP_INSTALL_CELL,
        CONFIG_TEMPLATE,
        *PIPELINE_CELLS,
        PREFLIGHT_CELL,
        RUN_CELL,
    ]
    return hashlib.sha256("\n\n".join(parts).encode("utf-8")).hexdigest()


def varint_literal(value):
    return repr(value)


def config_cell(name, switches):
    replacements = {
        "__VARIANT_NAME__": varint_literal(name),
        "__ABLATION_SWITCHES__": varint_literal(switches),
        "__ALL_VARIANTS__": varint_literal(ALL_VARIANTS),
        "__EXPECTED_VARIANTS__": varint_literal(VARIANTS),
        "__RAW_REPO__": varint_literal(SHARED_CONFIG["raw_repo"]),
        "__RAW_COMMIT_SHA__": varint_literal(SHARED_CONFIG["raw_commit_sha"]),
        "__OUTPUT_REPO__": varint_literal(SHARED_CONFIG["output_repo"]),
        "__OUTPUT_REVISION__": varint_literal(SHARED_CONFIG["output_revision"]),
        "__STUDY_ROOT__": varint_literal(SHARED_CONFIG["study_root"]),
        "__SOURCE_CSV_NAME__": varint_literal(SHARED_CONFIG["source_csv_name"]),
        "__COHORT_START__": varint_literal(SHARED_CONFIG["cohort_start"]),
        "__COHORT_STOP__": varint_literal(SHARED_CONFIG["cohort_stop"]),
        "__WORKERS__": varint_literal(SHARED_CONFIG["workers"]),
        "__PIPELINE_SHA256__": varint_literal(pipeline_sha256()),
        "__TEMPLATE_SHA256__": varint_literal(template_sha256()),
        "__GENERATED_NOTEBOOK_SHA256__": varint_literal(GENERATED_SHA_PLACEHOLDER),
    }
    text = CONFIG_TEMPLATE
    for placeholder, value in replacements.items():
        text = text.replace(placeholder, value)
    return text


def title_markdown(name, switches):
    return (
        f"# Ablation run: `{name}`\n\n"
        "Self-contained candidate-detection ablation notebook. "
        "The configured ablation switches are listed below.\n\n"
        f"- Switch set: `{json.dumps(switches, sort_keys=True)}`\n"
        f"- Fixed cohort: `df.iloc[{SHARED_CONFIG['cohort_start']}:{SHARED_CONFIG['cohort_stop']}]`\n"
        f"- Study root: `{SHARED_CONFIG['study_root']}`\n\n"
        "This notebook is generated by `ablation_generate_notebooks.ipynb`; edit the generator, not this file."
    )


def build_notebook(name):
    switches = VARIANTS[name]
    cells = [
        {"cell_type": "markdown", "metadata": {}, "source": title_markdown(name, switches)},
        {"cell_type": "markdown", "metadata": {}, "source": SETUP_MARKDOWN},
        {"cell_type": "code", "execution_count": None, "metadata": {}, "outputs": [], "source": PIP_INSTALL_CELL},
        {"cell_type": "code", "execution_count": None, "metadata": {}, "outputs": [], "source": config_cell(name, switches)},
    ]
    for source in PIPELINE_CELLS:
        cells.append({"cell_type": "code", "execution_count": None, "metadata": {}, "outputs": [], "source": source})
    cells.append({"cell_type": "code", "execution_count": None, "metadata": {}, "outputs": [], "source": PREFLIGHT_CELL})
    cells.append({"cell_type": "code", "execution_count": None, "metadata": {}, "outputs": [], "source": RUN_CELL})
    for index, cell in enumerate(cells, 1):
        cell["id"] = f"cell-{index}"
    notebook = {
        "cells": cells,
        "metadata": {
            "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
            "language_info": {"name": "python", "version": "3.10"},
        },
        "nbformat": 4,
        "nbformat_minor": 5,
    }
    canonical = json.dumps(notebook, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    digest = hashlib.sha256(canonical.encode("utf-8")).hexdigest()
    for cell in notebook["cells"]:
        cell["source"] = cell["source"].replace(GENERATED_SHA_PLACEHOLDER, digest)
    return notebook


def serialize_notebook(notebook):
    return json.dumps(notebook, indent=1, ensure_ascii=False) + "\n"


def notebook_path(name):
    return NOTEBOOK_DIR / f"ablation_{name}.ipynb"


def write_notebooks():
    for name in ALL_VARIANTS:
        path = notebook_path(name)
        path.write_text(serialize_notebook(build_notebook(name)), encoding="utf-8")
        print(f"wrote {path.name}")


def validate_notebooks():
    problems = []
    for name in ALL_VARIANTS:
        path = notebook_path(name)
        expected = build_notebook(name)
        if not path.is_file():
            problems.append(f"{path.name}: missing")
            continue
        try:
            actual = json.loads(path.read_text(encoding="utf-8"))
        except Exception as exc:
            problems.append(f"{path.name}: invalid JSON ({exc})")
            continue
        if actual != expected:
            if len(actual.get("cells", [])) != len(expected["cells"]):
                problems.append(f"{path.name}: cell count differs")
            else:
                for index, (left, right) in enumerate(zip(actual["cells"], expected["cells"]), 1):
                    if left != right:
                        problems.append(f"{path.name}: first drift in cell {index}")
                        break
            continue
        print(f"validated {path.name}")
    if problems:
        raise RuntimeError("Notebook drift or missing generated notebooks: " + "; ".join(problems))


if CHECK_ONLY:
    validate_notebooks()
else:
    if WRITE_NOTEBOOKS:
        write_notebooks()
    validate_notebooks()
    print("All six generated run notebooks match the canonical generator output.")


wrote ablation_baseline.ipynb
wrote ablation_optical_only.ipynb
wrote ablation_sar_only.ipynb
wrote ablation_no_slope.ipynb
wrote ablation_no_slic.ipynb
wrote ablation_no_bonferroni.ipynb
validated ablation_baseline.ipynb
validated ablation_optical_only.ipynb
validated ablation_sar_only.ipynb
validated ablation_no_slope.ipynb
validated ablation_no_slic.ipynb
validated ablation_no_bonferroni.ipynb
All six generated run notebooks match the canonical generator output.
